# Mediterranean Sea Surface Chlorophyll — Climatology, Anomalies & Extremes (1999–2025)

**Workflow**
1. Open the Mediterranean Sea BGC Reanalysis (`cmems_mod_med_bgc-plankton_my_4.2km_P1D-m`) — daily means, lazy (dask).
2. Resample daily → monthly means.
3. Compute the **yearly mean chlorophyll** at every pixel and explore it with a year slider.
4. Compute the **long-term climatological mean** at every pixel (mean over 1999–2025).
5. Compute the **monthly chlorophyll anomaly** = monthly value − pixel climatological mean.
6. Visualise the monthly anomaly with an interactive time slider showing the current month.
7. At every pixel, find the **minimum** and **maximum** of the anomaly time-series.
8. Plot spatial maps of the min-anomaly and max-anomaly fields.

> **Credentials**: requires a `~/.netrc` entry for `auth.marine.copernicus.eu`.

In [ ]:
import cmocean
import copernicusmarine
import hvplot.xarray
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Mediterranean bounding box
MED_BBOX = dict(
    minimum_longitude=-6,
    maximum_longitude=42,
    minimum_latitude=30,
    maximum_latitude=47,
)

# The BGC reanalysis starts January 1999; request through end of 2025
TIME_START = "1999-01-01"
TIME_END   = "2025-12-31"

## 1 — Open the chlorophyll dataset (lazy)

In [ ]:
%%time
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_bgc-plankton_my_4.2km_P1M-m",
    variables=["chl"],
    start_datetime=TIME_START,
    end_datetime=TIME_END,
    **MED_BBOX,
)
ds

In [ ]:
# Surface chlorophyll: select depth nearest to 0 m — result is (time, latitude, longitude), still lazy
chl_monthly = ds["chl"].sel(depth=0, method="nearest")
chl_monthly.attrs["long_name"] = "Monthly Mean Chlorophyll"
chl_monthly.attrs["units"] = "mg m-3"
print(f"Monthly chlorophyll shape: {chl_monthly.dims} = {dict(zip(chl_monthly.dims, chl_monthly.shape))}")
chl_monthly

## 2 — Monthly dataset (no resampling needed)

The `P1M-m` product is already at monthly resolution, so `chl_monthly` is ready to use directly.

In [ ]:
# No resampling required — dataset is already monthly (P1M-m)

## 3 — Yearly mean chlorophyll with time slider

Resample monthly → annual, compute (≈ 27 frames × grid ≈ small), then display with a year scrubber.

In [ ]:
cluster_type = 'Coiled'

In [ ]:
if cluster_type == 'Coiled':
    import coiled
    cluster = coiled.Cluster(
        region="eu-central-1",
        arm=True,   # run on ARM to save energy & cost
        worker_vm_types=["t4g.large"],  # cheap, small ARM instances, 2cpus, 2GB RAM
        n_workers=30, 
        name = "Emma",
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        software='protocoast-notebook-arm',  # Conda environment name
        workspace='esip-lab',
        timeout=180   # leave cluster running for 3 min in case we want to use it again
    )

    client = cluster.get_client()

In [ ]:
%%time
chl_yearly = chl_monthly.resample(time="YE").mean().compute()

# Replace the datetime coordinate with plain integer years for a cleaner slider
years = chl_yearly.time.dt.year.values
chl_yearly = (
    chl_yearly
    .assign_coords(year=("time", years))
    .swap_dims({"time": "year"})
    .drop_vars("time")
)
chl_yearly.attrs["long_name"] = "Annual Mean Chlorophyll"
chl_yearly.attrs["units"] = "mg m-3"
print(f"Yearly chlorophyll shape: {chl_yearly.dims} = {dict(zip(chl_yearly.dims, chl_yearly.shape))}")
chl_yearly

In [ ]:
chl_yearly.hvplot(
    x="longitude",
    y="latitude",
    groupby="year",
    rasterize=True,
    geo=True,
    cmap=cmocean.cm.algae,
    clabel="Chlorophyll (mg m⁻³)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title="Mediterranean Annual Mean Chlorophyll — use slider to step through years",
    fontscale=1.2,
    widget_type="scrubber",
    widget_location="bottom",
)

## 4 — Long-term climatological mean

Average every pixel over the full record (1999–2025). This is the reference field for anomaly computation.

In [ ]:
%%time
chl_clim = chl_monthly.mean(dim="time").compute()
chl_clim.attrs["long_name"] = f"Climatological Mean Chlorophyll ({TIME_START[:4]}\u2013{TIME_END[:4]})"
chl_clim.attrs["units"] = "mg m-3"
print(f"Chlorophyll clim range: {float(chl_clim.min()):.4f} \u2013 {float(chl_clim.max()):.4f} mg m\u207b\u00b3")
chl_clim

In [ ]:
chl_clim.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap=cmocean.cm.algae,
    clabel="Chlorophyll (mg m⁻³)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Climatological Mean Chlorophyll ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)

## 5 — Monthly chlorophyll anomaly

Anomaly = monthly chlorophyll − pixel climatological mean.

The subtraction broadcasts `chl_clim` (2-D) over the time axis of `chl_monthly` (3-D). The result remains a lazy dask array — no data is loaded into RAM until we ask for an explicit reduction.

In [ ]:
chl_anom = chl_monthly - chl_clim
chl_anom.attrs["long_name"] = "Chlorophyll Anomaly"
chl_anom.attrs["units"] = "mg m-3"
print(f"Anomaly shape: {chl_anom.dims} = {dict(zip(chl_anom.dims, chl_anom.shape))}")
chl_anom

## 5b — Monthly chlorophyll anomaly map with time slider

In [ ]:
import pandas as pd
import panel as pn

pn.extension()

# Load the full monthly anomaly into memory — needed for a responsive time slider
chl_anom_loaded = chl_anom.compute()

# Symmetric color limits so negative/positive departures are balanced around zero
anom_max = float(max(abs(chl_anom_loaded.min()), abs(chl_anom_loaded.max())))
times = chl_anom_loaded.time.values

# Integer-index Player widget (play/pause/step controls + scrubber)
player = pn.widgets.Player(
    start=0, end=len(times) - 1, value=0,
    loop_policy="loop", width=800, name="",
)

def plot_chl_frame(idx):
    label = pd.Timestamp(times[idx]).strftime("%B %Y")
    return chl_anom_loaded.isel(time=idx).hvplot(
        x="longitude",
        y="latitude",
        rasterize=True,
        geo=True,
        cmap="RdBu_r",
        clim=(-anom_max, anom_max),
        clabel="Chlorophyll Anomaly (mg m⁻³)",
        tiles="OSM",
        frame_width=800,
        frame_height=450,
        title=f"Mediterranean Monthly Chlorophyll Anomaly \u2014 {label}",
        fontscale=1.2,
    )

pn.Column(pn.bind(plot_chl_frame, player), player)

## 5c — Export monthly chlorophyll anomaly animation as MP4

Render one frame per month (1999–2025) of the **Mediterranean monthly chlorophyll anomaly** using matplotlib and save it as a downloadable MP4 video. Requires `ffmpeg` to be available on the system.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import pandas as pd

VIDEO_PATH = "med_chl_anomaly.mp4"

# chl_anom_loaded is already computed above (time × lat × lon)
lons = chl_anom_loaded.longitude.values
lats = chl_anom_loaded.latitude.values
times_list = chl_anom_loaded.time.values

# Symmetric color limits matching the interactive plot above
vmax = float(max(abs(np.nanmin(chl_anom_loaded.values)), abs(np.nanmax(chl_anom_loaded.values))))
vmin = -vmax

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor("#c8e6f5")

# First frame
data0 = chl_anom_loaded.isel(time=0).values
img = ax.pcolormesh(
    lons, lats, data0,
    cmap="RdBu_r",
    vmin=vmin, vmax=vmax,
    shading="auto",
)
cb = fig.colorbar(img, ax=ax, label="Chlorophyll Anomaly (mg m⁻³)", fraction=0.03, pad=0.02)
label0 = pd.Timestamp(times_list[0]).strftime("%B %Y")
title = ax.set_title(
    f"Mediterranean Monthly Chlorophyll Anomaly — {label0}",
    fontsize=14,
)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.tight_layout()

def update(frame_idx):
    data = chl_anom_loaded.isel(time=frame_idx).values
    img.set_array(data.ravel())
    label = pd.Timestamp(times_list[frame_idx]).strftime("%B %Y")
    title.set_text(f"Mediterranean Monthly Chlorophyll Anomaly — {label}")
    return img, title

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(times_list),
    interval=200,     # ms between frames
    blit=False,
)

ani.save(VIDEO_PATH, writer="ffmpeg", fps=5, dpi=120)
plt.close(fig)
print(f"Video saved → {VIDEO_PATH}")

In [ ]:
from IPython.display import FileLink, display
display(FileLink(VIDEO_PATH, result_html_prefix="Download video: "))

## 5d — Annual chlorophyll anomaly with year slider

Annual anomaly = yearly mean chlorophyll (from section 3) − long-term climatological mean (from section 4).

In [ ]:
import pandas as pd
import panel as pn

pn.extension()

# Annual anomaly: yearly mean − climatological mean (broadcasts over year dimension)
chl_annual_anom = chl_yearly - chl_clim
chl_annual_anom.attrs["long_name"] = "Annual Chlorophyll Anomaly"
chl_annual_anom.attrs["units"] = "mg m-3"

# Symmetric color limits
anom_yearly_max = float(max(abs(chl_annual_anom.min()), abs(chl_annual_anom.max())))
years_list = chl_annual_anom.year.values

player_yearly = pn.widgets.Player(
    start=0, end=len(years_list) - 1, value=0,
    loop_policy="loop", width=800, name="",
)

def plot_chl_annual_frame(idx):
    yr = int(years_list[idx])
    return chl_annual_anom.isel(year=idx).hvplot(
        x="longitude",
        y="latitude",
        rasterize=True,
        geo=True,
        cmap="RdBu_r",
        clim=(-anom_yearly_max, anom_yearly_max),
        clabel="Chlorophyll Anomaly (mg m⁻³)",
        tiles="OSM",
        frame_width=800,
        frame_height=450,
        title=f"Mediterranean Annual Chlorophyll Anomaly — {yr}",
        fontscale=1.2,
    )

pn.Column(pn.bind(plot_chl_annual_frame, player_yearly), player_yearly)

## 5e — Export annual chlorophyll anomaly animation as MP4

Render one frame per year (1999–2025) and save as a downloadable MP4. Requires `ffmpeg`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

VIDEO_PATH_ANNUAL = "med_chl_annual_anomaly.mp4"

lons = chl_annual_anom.longitude.values
lats = chl_annual_anom.latitude.values

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor("#c8e6f5")

data0 = chl_annual_anom.isel(year=0).values
img = ax.pcolormesh(
    lons, lats, data0,
    cmap="RdBu_r",
    vmin=-anom_yearly_max, vmax=anom_yearly_max,
    shading="auto",
)
fig.colorbar(img, ax=ax, label="Chlorophyll Anomaly (mg m⁻³)", fraction=0.03, pad=0.02)
title = ax.set_title(
    f"Mediterranean Annual Chlorophyll Anomaly — {int(years_list[0])}",
    fontsize=14,
)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.tight_layout()

def update_annual(frame_idx):
    data = chl_annual_anom.isel(year=frame_idx).values
    img.set_array(data.ravel())
    title.set_text(f"Mediterranean Annual Chlorophyll Anomaly — {int(years_list[frame_idx])}")
    return img, title

ani = animation.FuncAnimation(
    fig,
    update_annual,
    frames=len(years_list),
    interval=400,
    blit=False,
)

ani.save(VIDEO_PATH_ANNUAL, writer="ffmpeg", fps=3, dpi=120)
plt.close(fig)
print(f"Video saved → {VIDEO_PATH_ANNUAL}")

In [ ]:
from IPython.display import FileLink, display
display(FileLink(VIDEO_PATH_ANNUAL, result_html_prefix="Download video: "))

## 6 — Pixel-wise minimum and maximum anomaly

Collapse the time dimension by taking `min` and `max`. Both reductions are fused into a single dask graph pass over the data.

In [ ]:
%%time
import dask

# Compute both reductions in one scheduler call to share the I/O
chl_anom_min, chl_anom_max = dask.compute(
    chl_anom.min(dim="time"),
    chl_anom.max(dim="time"),
)

chl_anom_min.attrs["long_name"] = "Minimum Chlorophyll Anomaly"
chl_anom_min.attrs["units"] = "mg m-3"
chl_anom_max.attrs["long_name"] = "Maximum Chlorophyll Anomaly"
chl_anom_max.attrs["units"] = "mg m-3"

print(f"Min anomaly range : {float(chl_anom_min.min()):.4f} \u2013 {float(chl_anom_min.max()):.4f} mg m\u207b\u00b3")
print(f"Max anomaly range : {float(chl_anom_max.min()):.4f} \u2013 {float(chl_anom_max.max()):.4f} mg m\u207b\u00b3")

## 7 — Spatial maps of min and max anomaly

A diverging colormap (`RdBu_r`) centred on zero makes bloom-depleted/bloom-enhanced departures immediately readable.

In [ ]:
# Shared symmetric color limits so both maps are directly comparable
anom_abs_max = max(
    abs(float(chl_anom_min.min())),
    abs(float(chl_anom_max.max())),
)
clim_anom = (-anom_abs_max, anom_abs_max)
print(f"Symmetric color range: {clim_anom[0]:.4f} \u2013 {clim_anom[1]:.4f} mg m\u207b\u00b3")

In [ ]:
plot_min = chl_anom_min.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Chlorophyll Anomaly (mg m⁻³)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Minimum Chlorophyll Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_min

In [ ]:
plot_max = chl_anom_max.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Chlorophyll Anomaly (mg m⁻³)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Maximum Chlorophyll Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_max

## 8 — Side-by-side comparison

In [ ]:
plot_min_small = chl_anom_min.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Chlorophyll Anomaly (mg m⁻³)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Min Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

plot_max_small = chl_anom_max.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Chlorophyll Anomaly (mg m⁻³)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Max Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

(plot_min_small + plot_max_small).cols(2)